In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
actual_consumption = pd.read_csv('Actual_consumption_202301010000_202503050000_Quarterhour.csv', on_bad_lines='skip',delimiter=';')

In [3]:
actual_consumption

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions
0,"Jan 1, 2023 12:00 AM","Jan 1, 2023 12:15 AM","9,673.00","1,842.50",494.00
1,"Jan 1, 2023 12:15 AM","Jan 1, 2023 12:30 AM","9,593.50","1,691.50",502.50
2,"Jan 1, 2023 12:30 AM","Jan 1, 2023 12:45 AM","9,562.00","1,442.50",561.00
3,"Jan 1, 2023 12:45 AM","Jan 1, 2023 1:00 AM","9,517.50","1,598.50",519.25
4,"Jan 1, 2023 1:00 AM","Jan 1, 2023 1:15 AM","9,433.25","1,325.50",301.00
...,...,...,...,...,...
76219,"Mar 4, 2025 10:45 PM","Mar 4, 2025 11:00 PM",-,-,-
76220,"Mar 4, 2025 11:00 PM","Mar 4, 2025 11:15 PM",-,-,-
76221,"Mar 4, 2025 11:15 PM","Mar 4, 2025 11:30 PM",-,-,-
76222,"Mar 4, 2025 11:30 PM","Mar 4, 2025 11:45 PM",-,-,-


In [4]:
actual_consumption['Start date'] = pd.to_datetime(actual_consumption['Start date'], format= "%b %d, %Y %I:%M %p")
actual_consumption['End date'] = pd.to_datetime(actual_consumption['End date'], format= "%b %d, %Y %I:%M %p")

In [5]:
actual_consumption['hour'] = actual_consumption['Start date'].dt.hour
actual_consumption['day'] = actual_consumption['Start date'].dt.day
actual_consumption['weekday'] = actual_consumption['Start date'].dt.weekday
actual_consumption['week'] = actual_consumption['Start date'].dt.isocalendar().week
actual_consumption['month'] = actual_consumption['Start date'].dt.month
actual_consumption['year'] = actual_consumption['Start date'].dt.year

In [6]:
actual_consumption['Total (grid load) [MWh] Original resolutions'] = actual_consumption['Total (grid load) [MWh] Original resolutions'].replace({',': '', '-': None}, regex=True)
actual_consumption['Residual load [MWh] Original resolutions'] = actual_consumption['Residual load [MWh] Original resolutions'].replace({',': '', '-': None}, regex=True)
actual_consumption['Hydro pumped storage [MWh] Original resolutions'] = actual_consumption['Hydro pumped storage [MWh] Original resolutions'].replace({',': '', '-': None}, regex=True)

In [7]:
actual_consumption['Total (grid load) [MWh] Original resolutions'] = pd.to_numeric(actual_consumption['Total (grid load) [MWh] Original resolutions'], errors='coerce')
actual_consumption['Residual load [MWh] Original resolutions'] = pd.to_numeric(actual_consumption['Residual load [MWh] Original resolutions'], errors='coerce')
actual_consumption['Hydro pumped storage [MWh] Original resolutions'] = pd.to_numeric(actual_consumption['Hydro pumped storage [MWh] Original resolutions'], errors='coerce')

In [8]:
actual_consumption.head()

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions,hour,day,weekday,week,month,year
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,2023
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,2023
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,2023
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,2023
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,2023


In [9]:
actual_consumption.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76224 entries, 0 to 76223
Data columns (total 11 columns):
 #   Column                                           Non-Null Count  Dtype         
---  ------                                           --------------  -----         
 0   Start date                                       76224 non-null  datetime64[ns]
 1   End date                                         76224 non-null  datetime64[ns]
 2   Total (grid load) [MWh] Original resolutions     76207 non-null  float64       
 3   Residual load [MWh] Original resolutions         75670 non-null  float64       
 4   Hydro pumped storage [MWh] Original resolutions  76207 non-null  float64       
 5   hour                                             76224 non-null  int32         
 6   day                                              76224 non-null  int32         
 7   weekday                                          76224 non-null  int32         
 8   week                                

In [10]:
actual_generation = pd.read_csv('Actual_generation_202301010000_202503050000_Quarterhour.csv', on_bad_lines='skip', delimiter=';', low_memory=False)

In [11]:
actual_generation.head()

,Start date,End date,Biomass [MWh] Original resolutions,Hydropower [MWh] Original resolutions,Wind offshore [MWh] Original resolutions,Wind onshore [MWh] Original resolutions,Photovoltaics [MWh] Original resolutions,Other renewable [MWh] Original resolutions,Nuclear [MWh] Original resolutions,Lignite [MWh] Original resolutions,Hard coal [MWh] Original resolutions,Fossil gas [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions,Other conventional [MWh] Original resolutions
0,"Jan 1, 2023 12:00 AM","Jan 1, 2023 12:15 AM","1,006.25",319.75,684.25,"7,145.75",0.50,32.25,615.25,962.75,517.00,453.75,31.75,307.25
1,"Jan 1, 2023 12:15 AM","Jan 1, 2023 12:30 AM","1,003.50",317.25,743.50,"7,158.25",0.25,32.25,614.75,963.25,518.00,453.50,45.00,307.25
2,"Jan 1, 2023 12:30 AM","Jan 1, 2023 12:45 AM","1,003.00",317.00,817.00,"7,302.25",0.25,32.50,615.00,966.50,517.00,456.00,26.75,308.25
3,"Jan 1, 2023 12:45 AM","Jan 1, 2023 1:00 AM","1,001.50",321.25,814.50,"7,104.25",0.25,32.50,614.50,966.75,515.50,454.50,21.75,306.00
4,"Jan 1, 2023 1:00 AM","Jan 1, 2023 1:15 AM",997.50,314.75,785.50,"7,322.00",0.25,32.25,614.50,969.00,513.25,415.00,137.00,306.75


In [12]:
columns = actual_generation.columns[2:]

In [13]:
actual_generation[columns] = actual_generation[columns].replace({'-': None, ',': ''}, regex=True)

In [14]:
for col in columns:
    actual_generation[col] = pd.to_numeric(actual_generation[col], errors='coerce') 

In [15]:
actual_generation['Start date'] = pd.to_datetime(actual_generation['Start date'], format='%b %d, %Y %I:%M %p')
actual_generation['End date'] = pd.to_datetime(actual_generation['End date'], format='%b %d, %Y %I:%M %p')

In [16]:
actual_generation['hour'] = actual_generation['Start date'].dt.hour
actual_generation['day'] = actual_generation['Start date'].dt.day
actual_generation['weekday'] = actual_generation['Start date'].dt.weekday
actual_generation['week'] = actual_generation['Start date'].dt.isocalendar().week
actual_generation['month'] = actual_generation['Start date'].dt.month
actual_generation['year'] = actual_generation['Start date'].dt.year

In [17]:
columns2 = actual_generation.columns[2:14]

In [18]:
actual_generation['Total Generation [MWh]'] = actual_generation[columns2].sum(axis=1)
actual_generation.head()

,Start date,End date,Biomass [MWh] Original resolutions,Hydropower [MWh] Original resolutions,Wind offshore [MWh] Original resolutions,Wind onshore [MWh] Original resolutions,Photovoltaics [MWh] Original resolutions,Other renewable [MWh] Original resolutions,Nuclear [MWh] Original resolutions,Lignite [MWh] Original resolutions,...,Fossil gas [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions,Other conventional [MWh] Original resolutions,hour,day,weekday,week,month,year,Total Generation [MWh]
0,2023-01-01 00:00:00,2023-01-01 00:15:00,1006.25,319.75,684.25,7145.75,0.50,32.25,615.25,962.75,...,453.75,31.75,307.25,0,1,6,52,1,2023,12076.50
1,2023-01-01 00:15:00,2023-01-01 00:30:00,1003.50,317.25,743.50,7158.25,0.25,32.25,614.75,963.25,...,453.50,45.00,307.25,0,1,6,52,1,2023,12156.75
2,2023-01-01 00:30:00,2023-01-01 00:45:00,1003.00,317.00,817.00,7302.25,0.25,32.50,615.00,966.50,...,456.00,26.75,308.25,0,1,6,52,1,2023,12361.50
3,2023-01-01 00:45:00,2023-01-01 01:00:00,1001.50,321.25,814.50,7104.25,0.25,32.50,614.50,966.75,...,454.50,21.75,306.00,0,1,6,52,1,2023,12153.25
4,2023-01-01 01:00:00,2023-01-01 01:15:00,997.50,314.75,785.50,7322.00,0.25,32.25,614.50,969.00,...,415.00,137.00,306.75,1,1,6,52,1,2023,12407.75


In [19]:
merged_df = pd.merge(actual_consumption, actual_generation, on=['Start date', 'End date', 'hour', 'day', 'weekday', 'week', 'month', 'year'],how='inner')
merged_df.head()

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Wind onshore [MWh] Original resolutions,Photovoltaics [MWh] Original resolutions,Other renewable [MWh] Original resolutions,Nuclear [MWh] Original resolutions,Lignite [MWh] Original resolutions,Hard coal [MWh] Original resolutions,Fossil gas [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_y,Other conventional [MWh] Original resolutions,Total Generation [MWh]
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,7145.75,0.50,32.25,615.25,962.75,517.00,453.75,31.75,307.25,12076.50
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,7158.25,0.25,32.25,614.75,963.25,518.00,453.50,45.00,307.25,12156.75
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,7302.25,0.25,32.50,615.00,966.50,517.00,456.00,26.75,308.25,12361.50
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,7104.25,0.25,32.50,614.50,966.75,515.50,454.50,21.75,306.00,12153.25
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,7322.00,0.25,32.25,614.50,969.00,513.25,415.00,137.00,306.75,12407.75


In [20]:
automatice_frequency = pd.read_csv('Automatic_Frequency_Restoration_Reserve_202301010000_202503050000_Quarterhour.csv',on_bad_lines='skip', delimiter=';', low_memory=False)

In [21]:
automatice_frequency.drop(columns=['Activation price (+) [€/MWh] Original resolutions','Activation price (-) [€/MWh] Original resolutions'],inplace=True)

In [22]:
automatice_frequency['Start date'] = pd.to_datetime(automatice_frequency['Start date'], format='%b %d, %Y %I:%M %p')
automatice_frequency['End date'] = pd.to_datetime(automatice_frequency['End date'], format='%b %d, %Y %I:%M %p')

In [23]:
automatice_frequency['hour'] = automatice_frequency['Start date'].dt.hour
automatice_frequency['day'] = automatice_frequency['Start date'].dt.day
automatice_frequency['weekday'] = automatice_frequency['Start date'].dt.weekday
automatice_frequency['week'] = automatice_frequency['Start date'].dt.isocalendar().week
automatice_frequency['month'] = automatice_frequency['Start date'].dt.month
automatice_frequency['year'] = automatice_frequency['Start date'].dt.year

In [24]:
automatice_frequency.head()

,Start date,End date,Volume activated (+) [MWh] Original resolutions,Volume activated (-) [MWh] Original resolutions,Volume procured (+) [MW] Original resolutions,Volume procured (-) [MW] Original resolutions,Procurement price (+) [€/MW] Original resolutions,Procurement price (-) [€/MW] Original resolutions,hour,day,weekday,week,month,year
0,2023-01-01 00:00:00,2023-01-01 00:15:00,0.00,15.50,"1,941.00","2,029.00",0.52,7.52,0,1,6,52,1,2023
1,2023-01-01 00:15:00,2023-01-01 00:30:00,20.75,41.75,"1,941.00","2,029.00",0.52,7.52,0,1,6,52,1,2023
2,2023-01-01 00:30:00,2023-01-01 00:45:00,0.00,82.50,"1,941.00","2,029.00",0.52,7.52,0,1,6,52,1,2023
3,2023-01-01 00:45:00,2023-01-01 01:00:00,19.00,11.25,"1,941.00","2,029.00",0.52,7.52,0,1,6,52,1,2023
4,2023-01-01 01:00:00,2023-01-01 01:15:00,155.00,0.00,"1,941.00","2,029.00",0.52,7.52,1,1,6,52,1,2023


In [25]:
col = automatice_frequency.columns[2:8]

In [26]:
automatice_frequency[col] = automatice_frequency[col].replace({'-': None, ',': ''}, regex=True).apply(pd.to_numeric, errors='coerce')

In [27]:
automatice_frequency.head()

,Start date,End date,Volume activated (+) [MWh] Original resolutions,Volume activated (-) [MWh] Original resolutions,Volume procured (+) [MW] Original resolutions,Volume procured (-) [MW] Original resolutions,Procurement price (+) [€/MW] Original resolutions,Procurement price (-) [€/MW] Original resolutions,hour,day,weekday,week,month,year
0,2023-01-01 00:00:00,2023-01-01 00:15:00,0.00,15.50,1941.0,2029.0,0.52,7.52,0,1,6,52,1,2023
1,2023-01-01 00:15:00,2023-01-01 00:30:00,20.75,41.75,1941.0,2029.0,0.52,7.52,0,1,6,52,1,2023
2,2023-01-01 00:30:00,2023-01-01 00:45:00,0.00,82.50,1941.0,2029.0,0.52,7.52,0,1,6,52,1,2023
3,2023-01-01 00:45:00,2023-01-01 01:00:00,19.00,11.25,1941.0,2029.0,0.52,7.52,0,1,6,52,1,2023
4,2023-01-01 01:00:00,2023-01-01 01:15:00,155.00,0.00,1941.0,2029.0,0.52,7.52,1,1,6,52,1,2023


In [28]:
automatice_frequency.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76224 entries, 0 to 76223
Data columns (total 14 columns):
 #   Column                                             Non-Null Count  Dtype         
---  ------                                             --------------  -----         
 0   Start date                                         76224 non-null  datetime64[ns]
 1   End date                                           76224 non-null  datetime64[ns]
 2   Volume activated (+) [MWh] Original resolutions    30012 non-null  float64       
 3   Volume activated (-) [MWh] Original resolutions    30012 non-null  float64       
 4   Volume procured (+) [MW] Original resolutions      17372 non-null  float64       
 5   Volume procured (-) [MW] Original resolutions      17372 non-null  float64       
 6   Procurement price (+) [€/MW] Original resolutions  17372 non-null  float64       
 7   Procurement price (-) [€/MW] Original resolutions  17372 non-null  float64       
 8   hour            

In [29]:
merged_df2 = pd.merge(merged_df, automatice_frequency, on=['Start date', 'End date', 'hour', 'day', 'weekday', 'week', 'month', 'year'], how='inner')
merged_df2.head()

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Fossil gas [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_y,Other conventional [MWh] Original resolutions,Total Generation [MWh],Volume activated (+) [MWh] Original resolutions,Volume activated (-) [MWh] Original resolutions,Volume procured (+) [MW] Original resolutions,Volume procured (-) [MW] Original resolutions,Procurement price (+) [€/MW] Original resolutions,Procurement price (-) [€/MW] Original resolutions
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,453.75,31.75,307.25,12076.50,0.00,15.50,1941.0,2029.0,0.52,7.52
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,453.50,45.00,307.25,12156.75,20.75,41.75,1941.0,2029.0,0.52,7.52
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,456.00,26.75,308.25,12361.50,0.00,82.50,1941.0,2029.0,0.52,7.52
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,454.50,21.75,306.00,12153.25,19.00,11.25,1941.0,2029.0,0.52,7.52
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,415.00,137.00,306.75,12407.75,155.00,0.00,1941.0,2029.0,0.52,7.52


In [30]:
balancing_energy = pd.read_csv('Balancing_energy_202301010000_202503050000_Quarterhour_Month.csv', on_bad_lines='skip',delimiter=';', low_memory=False)

In [31]:
balancing_energy['Start date'] = pd.to_datetime(balancing_energy['Start date'], format='%b %d, %Y %I:%M %p')
balancing_energy['End date'] = pd.to_datetime(balancing_energy['End date'], format='%b %d, %Y %I:%M %p')

In [32]:
balancing_energy.drop(columns=['Volume (+) [MWh] Original resolutions','Volume (-) [MWh] Original resolutions'], inplace=True)


In [33]:
balancing_energy.head()

,Start date,End date,Price [€/MWh] Original resolutions,Net income [€] Original resolutions
0,2023-01-01 00:00:00,2023-01-01 00:15:00,185.56,"31,476,878.85"
1,2023-01-01 00:15:00,2023-01-01 00:30:00,-78.96,-
2,2023-01-01 00:30:00,2023-01-01 00:45:00,-76.74,-
3,2023-01-01 00:45:00,2023-01-01 01:00:00,-60.71,-
4,2023-01-01 01:00:00,2023-01-01 01:15:00,250.99,-


In [34]:
col = balancing_energy.columns[2:4]

In [35]:
balancing_energy[col] = balancing_energy[col].replace({'^-$': np.nan, ',': ''}, regex=True).apply(pd.to_numeric, errors='coerce')

In [36]:
balancing_energy['hour'] = balancing_energy['Start date'].dt.hour
balancing_energy['day'] = balancing_energy['Start date'].dt.day
balancing_energy['week'] = balancing_energy['Start date'].dt.isocalendar().week
balancing_energy['weekday'] = balancing_energy['Start date'].dt.weekday  
balancing_energy['month'] = balancing_energy['Start date'].dt.month
balancing_energy['year'] = balancing_energy['Start date'].dt.year

In [37]:
balancing_energy.head()

,Start date,End date,Price [€/MWh] Original resolutions,Net income [€] Original resolutions,hour,day,week,weekday,month,year
0,2023-01-01 00:00:00,2023-01-01 00:15:00,185.56,31476878.85,0,1,52,6,1,2023
1,2023-01-01 00:15:00,2023-01-01 00:30:00,-78.96,NaN,0,1,52,6,1,2023
2,2023-01-01 00:30:00,2023-01-01 00:45:00,-76.74,NaN,0,1,52,6,1,2023
3,2023-01-01 00:45:00,2023-01-01 01:00:00,-60.71,NaN,0,1,52,6,1,2023
4,2023-01-01 01:00:00,2023-01-01 01:15:00,250.99,NaN,1,1,52,6,1,2023


In [38]:
merged_df3 = pd.merge(
    merged_df2, 
    balancing_energy, 
    on=['Start date', 'End date', 'hour', 'day', 'weekday', 'week', 'month', 'year'], 
    how='inner'
)
merged_df3.head()

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Other conventional [MWh] Original resolutions,Total Generation [MWh],Volume activated (+) [MWh] Original resolutions,Volume activated (-) [MWh] Original resolutions,Volume procured (+) [MW] Original resolutions,Volume procured (-) [MW] Original resolutions,Procurement price (+) [€/MW] Original resolutions,Procurement price (-) [€/MW] Original resolutions,Price [€/MWh] Original resolutions,Net income [€] Original resolutions
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,307.25,12076.50,0.00,15.50,1941.0,2029.0,0.52,7.52,185.56,31476878.85
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,307.25,12156.75,20.75,41.75,1941.0,2029.0,0.52,7.52,-78.96,NaN
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,308.25,12361.50,0.00,82.50,1941.0,2029.0,0.52,7.52,-76.74,NaN
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,306.00,12153.25,19.00,11.25,1941.0,2029.0,0.52,7.52,-60.71,NaN
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,306.75,12407.75,155.00,0.00,1941.0,2029.0,0.52,7.52,250.99,NaN


In [39]:
costs = pd.read_csv('Costs_of_TSOs__without_costs_of_DSOs__202301010000_202503050000_Month.csv', on_bad_lines='skip', delimiter=';', low_memory=False)

In [40]:
costs['Start date'] = pd.to_datetime(costs['Start date'], format='%b %d, %Y')
costs['End date'] = pd.to_datetime(costs['End date'], format='%b %d, %Y')

In [41]:
costs['hour'] = costs['Start date'].dt.hour
costs['day'] = costs['Start date'].dt.day
costs['week'] = costs['Start date'].dt.isocalendar().week
costs['weekday'] = costs['Start date'].dt.weekday 
costs['month'] = costs['Start date'].dt.month
costs['year'] = costs['Start date'].dt.year

In [42]:
col = costs.columns[2:5]

In [43]:
costs[col] = costs[col].replace({'-': None, ',': ''}, regex=True).apply(pd.to_numeric, errors='coerce')

In [44]:
costs.head()

,Start date,End date,Balancing services [€] Original resolutions,Network security of the TSOs [€] Original resolutions,Countertrading [€] Original resolutions,hour,day,week,weekday,month,year
0,2023-01-01,2023-02-01,71578546.31,3.982759e+08,17519420.07,0,1,52,6,1,2023
1,2023-02-01,2023-03-01,40835671.81,3.362633e+08,19384916.81,0,1,5,2,2,2023
2,2023-03-01,2023-04-01,71751158.39,2.998916e+08,16184262.39,0,1,9,2,3,2023
3,2023-04-01,2023-05-01,82581492.42,2.170505e+08,17818622.42,0,1,13,5,4,2023
4,2023-05-01,2023-06-01,61944912.17,1.278177e+08,18333011.12,0,1,18,0,5,2023


In [45]:
merged_df4 = pd.merge(
    merged_df3,
    costs,
    on=['hour', 'day', 'weekday', 'week', 'month', 'year'], 
    how='left'
)
merged_df4.head()


,Start date_x,End date_x,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Volume procured (-) [MW] Original resolutions,Procurement price (+) [€/MW] Original resolutions,Procurement price (-) [€/MW] Original resolutions,Price [€/MWh] Original resolutions,Net income [€] Original resolutions,Start date_y,End date_y,Balancing services [€] Original resolutions,Network security of the TSOs [€] Original resolutions,Countertrading [€] Original resolutions
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,2029.0,0.52,7.52,185.56,31476878.85,2023-01-01,2023-02-01,71578546.31,3.982759e+08,17519420.07
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,2029.0,0.52,7.52,-78.96,NaN,2023-01-01,2023-02-01,71578546.31,3.982759e+08,17519420.07
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,2029.0,0.52,7.52,-76.74,NaN,2023-01-01,2023-02-01,71578546.31,3.982759e+08,17519420.07
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,2029.0,0.52,7.52,-60.71,NaN,2023-01-01,2023-02-01,71578546.31,3.982759e+08,17519420.07
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,2029.0,0.52,7.52,250.99,NaN,NaT,NaT,NaN,NaN,NaN


In [46]:
merged_df4.rename(columns={'Start date_x': 'Start date', 'End date_x': 'End date'}, inplace=True)

In [47]:
merged_df4.drop(columns=['Start date_y', 'End date_y'], inplace=True)
merged_df4.head()

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Volume activated (-) [MWh] Original resolutions,Volume procured (+) [MW] Original resolutions,Volume procured (-) [MW] Original resolutions,Procurement price (+) [€/MW] Original resolutions,Procurement price (-) [€/MW] Original resolutions,Price [€/MWh] Original resolutions,Net income [€] Original resolutions,Balancing services [€] Original resolutions,Network security of the TSOs [€] Original resolutions,Countertrading [€] Original resolutions
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,15.50,1941.0,2029.0,0.52,7.52,185.56,31476878.85,71578546.31,3.982759e+08,17519420.07
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,41.75,1941.0,2029.0,0.52,7.52,-78.96,NaN,71578546.31,3.982759e+08,17519420.07
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,82.50,1941.0,2029.0,0.52,7.52,-76.74,NaN,71578546.31,3.982759e+08,17519420.07
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,11.25,1941.0,2029.0,0.52,7.52,-60.71,NaN,71578546.31,3.982759e+08,17519420.07
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,0.00,1941.0,2029.0,0.52,7.52,250.99,NaN,NaN,NaN,NaN


In [48]:
cross_border = pd.read_csv('Cross-border_physical_flows_202301010000_202503050000_Quarterhour.csv',on_bad_lines='skip',delimiter=';', low_memory=False)

In [49]:
cross_border['Start date'] = pd.to_datetime(cross_border['Start date'], format='%b %d, %Y %I:%M %p')
cross_border['End date'] = pd.to_datetime(cross_border['End date'], format='%b %d, %Y %I:%M %p')

In [50]:
cross_border['hour'] = cross_border['Start date'].dt.hour
cross_border['day'] = cross_border['Start date'].dt.day
cross_border['week'] = cross_border['Start date'].dt.isocalendar().week
cross_border['weekday'] = cross_border['Start date'].dt.weekday
cross_border['month'] = cross_border['Start date'].dt.month
cross_border['year'] = cross_border['Start date'].dt.year

In [51]:
cross_border.head()

,Start date,End date,Net export [MWh] Original resolutions,Netherlands (export) [MWh] Original resolutions,Netherlands (import) [MWh] Original resolutions,Switzerland (export) [MWh] Original resolutions,Switzerland (import) [MWh] Original resolutions,Denmark (export) [MWh] Original resolutions,Denmark (import) [MWh] Original resolutions,Czech Republic (export) [MWh] Original resolutions,...,Norway (export) [MWh] Original resolutions,Norway (import) [MWh] Original resolutions,Belgium (export) [MWh] Original resolutions,Belgium (import) [MWh] Original resolutions,hour,day,week,weekday,month,year
0,2023-01-01 00:00:00,2023-01-01 00:15:00,"3,013.75",226.00,-186.75,545.00,0.00,691.00,0.00,348.25,...,306.75,0.00,46.25,0.00,0,1,52,6,1,2023
1,2023-01-01 00:15:00,2023-01-01 00:30:00,"3,011.50",294.25,-259.75,545.50,0.00,653.50,0.00,348.25,...,322.00,0.00,42.00,0.00,0,1,52,6,1,2023
2,2023-01-01 00:30:00,2023-01-01 00:45:00,"2,984.00",299.50,-260.00,524.75,0.00,636.75,-0.25,348.25,...,322.25,0.00,53.75,0.00,0,1,52,6,1,2023
3,2023-01-01 00:45:00,2023-01-01 01:00:00,"2,992.50",295.75,-225.25,520.00,-2.25,625.00,-0.25,348.25,...,322.25,0.00,34.50,0.00,0,1,52,6,1,2023
4,2023-01-01 01:00:00,2023-01-01 01:15:00,"3,637.00",343.50,-5.25,629.75,-24.00,599.50,0.00,385.00,...,322.25,0.00,0.00,-122.00,1,1,52,6,1,2023


In [52]:
columns = [
    'Net export [MWh] Original resolutions',
    'Netherlands (export) [MWh] Original resolutions', 'Netherlands (import) [MWh] Original resolutions',
    'Switzerland (export) [MWh] Original resolutions', 'Switzerland (import) [MWh] Original resolutions',
    'Denmark (export) [MWh] Original resolutions', 'Denmark (import) [MWh] Original resolutions',
    'Czech Republic (export) [MWh] Original resolutions', 'Czech Republic (import) [MWh] Original resolutions',
    'Luxembourg (export) [MWh] Original resolutions', 'Luxembourg (import) [MWh] Original resolutions',
    'Sweden (export) [MWh] Original resolutions', 'Sweden (import) [MWh] Original resolutions',
    'Austria (export) [MWh] Original resolutions', 'Austria (import) [MWh] Original resolutions',
    'France (export) [MWh] Original resolutions', 'France (import) [MWh] Original resolutions',
    'Poland (export) [MWh] Original resolutions', 'Poland (import) [MWh] Original resolutions',
    'Norway (export) [MWh] Original resolutions', 'Norway (import) [MWh] Original resolutions',
    'Belgium (export) [MWh] Original resolutions', 'Belgium (import) [MWh] Original resolutions'
]

def clean_numeric_column(series):
    return (
        series.astype(str)
        .str.replace(',', '', regex=False)
        .str.strip()  
        .replace(r'^\s*-$', np.nan, regex=True)  
        .astype(float)  )


cross_border[columns] = cross_border[columns].apply(clean_numeric_column)

In [53]:
merged_df5 = pd.merge(
    merged_df4, 
    cross_border, 
    on=['Start date', 'End date', 'hour', 'day', 'weekday', 'week', 'month', 'year'], 
    how='inner'
)

merged_df5.head()

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Austria (export) [MWh] Original resolutions,Austria (import) [MWh] Original resolutions,France (export) [MWh] Original resolutions,France (import) [MWh] Original resolutions,Poland (export) [MWh] Original resolutions,Poland (import) [MWh] Original resolutions,Norway (export) [MWh] Original resolutions,Norway (import) [MWh] Original resolutions,Belgium (export) [MWh] Original resolutions,Belgium (import) [MWh] Original resolutions
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,564.75,-25.50,215.25,-1.0,266.25,0.0,306.75,0.0,46.25,0.0
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,621.50,-29.50,215.25,-1.0,266.25,0.0,322.00,0.0,42.00,0.0
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,598.75,-12.25,215.25,-1.0,266.25,0.0,322.25,0.0,53.75,0.0
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,608.75,-12.50,215.25,-1.0,266.25,0.0,322.25,0.0,34.50,0.0
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,688.75,-8.25,502.00,0.0,293.25,0.0,322.25,0.0,0.00,-122.0


In [54]:
day_ahead =  pd.read_csv('Day-ahead_prices_202301010000_202503050000_Hour.csv',on_bad_lines='skip', delimiter=';', low_memory=False)

In [55]:
day_ahead.drop(columns=['DE/AT/LU [€/MWh] Original resolutions'],inplace=True)

In [56]:
columns = [
    'Northern Italy [€/MWh] Original resolutions',
    'Slovenia [€/MWh] Original resolutions'
]

day_ahead[columns] = day_ahead[columns].apply(pd.to_numeric, errors='coerce')

In [57]:
day_ahead['Start date'] = pd.to_datetime(day_ahead['Start date'], format='%b %d, %Y %I:%M %p')
day_ahead['End date'] = pd.to_datetime(day_ahead['End date'], format='%b %d, %Y %I:%M %p')

In [58]:
day_ahead['hour'] = day_ahead['Start date'].dt.hour
day_ahead['day'] = day_ahead['Start date'].dt.day
day_ahead['week'] = day_ahead['Start date'].dt.isocalendar().week
day_ahead['weekday'] = day_ahead['Start date'].dt.weekday 
day_ahead['month'] = day_ahead['Start date'].dt.month
day_ahead['year'] = day_ahead['Start date'].dt.year

In [59]:
day_ahead.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19056 entries, 0 to 19055
Data columns (total 24 columns):
 #   Column                                           Non-Null Count  Dtype         
---  ------                                           --------------  -----         
 0   Start date                                       19056 non-null  datetime64[ns]
 1   End date                                         19056 non-null  datetime64[ns]
 2   Germany/Luxembourg [€/MWh] Original resolutions  19056 non-null  float64       
 3   ∅ DE/LU neighbours [€/MWh] Original resolutions  19056 non-null  float64       
 4   Belgium [€/MWh] Original resolutions             19056 non-null  float64       
 5   Denmark 1 [€/MWh] Original resolutions           19056 non-null  float64       
 6   Denmark 2 [€/MWh] Original resolutions           19056 non-null  float64       
 7   France [€/MWh] Original resolutions              19056 non-null  float64       
 8   Netherlands [€/MWh] Original resolut

In [60]:
merged_df6 = pd.merge(
    merged_df5, 
    day_ahead, 
    on=['hour', 'day', 'weekday', 'week', 'month', 'year'], 
    how='left'
)
merged_df6.head()

,Start date_x,End date_x,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Netherlands [€/MWh] Original resolutions,Norway 2 [€/MWh] Original resolutions,Austria [€/MWh] Original resolutions,Poland [€/MWh] Original resolutions,Sweden 4 [€/MWh] Original resolutions,Switzerland [€/MWh] Original resolutions,Czech Republic [€/MWh] Original resolutions,Northern Italy [€/MWh] Original resolutions,Slovenia [€/MWh] Original resolutions,Hungary [€/MWh] Original resolutions
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,-3.61,119.32,12.06,18.09,2.01,0.03,4.84,195.90,13.31,19.76
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,-3.61,119.32,12.06,18.09,2.01,0.03,4.84,195.90,13.31,19.76
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,-3.61,119.32,12.06,18.09,2.01,0.03,4.84,195.90,13.31,19.76
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,-3.61,119.32,12.06,18.09,2.01,0.03,4.84,195.90,13.31,19.76
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,-1.46,108.83,-0.10,5.75,1.38,-7.25,-0.35,191.09,-0.07,0.19


In [61]:
merged_df6.rename(columns={'Start date_x': 'Start date', 'End date_x': 'End date'}, inplace=True)
merged_df6.drop(columns=['Start date_y', 'End date_y'], inplace=True)

In [62]:
exported_balance = pd.read_csv('Exported_balancing_services_202301010000_202503050000_Quarterhour.csv',on_bad_lines='skip', delimiter=';', low_memory=False)

In [63]:
exported_balance['Start date'] = pd.to_datetime(exported_balance['Start date'], format='%b %d, %Y %I:%M %p')
exported_balance['End date'] = pd.to_datetime(exported_balance['End date'], format='%b %d, %Y %I:%M %p')

In [64]:
exported_balance['Austria [MWh] Original resolutions'] = exported_balance['Austria [MWh] Original resolutions'].replace({'-': np.nan, ',': ''}, regex=True).apply(pd.to_numeric, errors='coerce')

In [65]:
exported_balance['hour'] = exported_balance['Start date'].dt.hour
exported_balance['day'] = exported_balance['Start date'].dt.day
exported_balance['week'] = exported_balance['Start date'].dt.isocalendar().week
exported_balance['weekday'] = exported_balance['Start date'].dt.weekday
exported_balance['month'] = exported_balance['Start date'].dt.month
exported_balance['year'] = exported_balance['Start date'].dt.year

In [66]:
exported_balance.head()

,Start date,End date,Austria [MWh] Original resolutions,hour,day,week,weekday,month,year
0,2023-01-01 00:00:00,2023-01-01 00:15:00,57.50,0,1,52,6,1,2023
1,2023-01-01 00:15:00,2023-01-01 00:30:00,23.25,0,1,52,6,1,2023
2,2023-01-01 00:30:00,2023-01-01 00:45:00,28.75,0,1,52,6,1,2023
3,2023-01-01 00:45:00,2023-01-01 01:00:00,22.25,0,1,52,6,1,2023
4,2023-01-01 01:00:00,2023-01-01 01:15:00,10.50,1,1,52,6,1,2023


In [67]:
merged_df7 = pd.merge(
    merged_df6, 
    exported_balance, 
    on=['Start date', 'End date', 'hour', 'day', 'weekday', 'week', 'month', 'year'], 
    how='inner'
)

merged_df7.head()

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Norway 2 [€/MWh] Original resolutions,Austria [€/MWh] Original resolutions,Poland [€/MWh] Original resolutions,Sweden 4 [€/MWh] Original resolutions,Switzerland [€/MWh] Original resolutions,Czech Republic [€/MWh] Original resolutions,Northern Italy [€/MWh] Original resolutions,Slovenia [€/MWh] Original resolutions,Hungary [€/MWh] Original resolutions,Austria [MWh] Original resolutions
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,119.32,12.06,18.09,2.01,0.03,4.84,195.90,13.31,19.76,57.50
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,119.32,12.06,18.09,2.01,0.03,4.84,195.90,13.31,19.76,23.25
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,119.32,12.06,18.09,2.01,0.03,4.84,195.90,13.31,19.76,28.75
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,119.32,12.06,18.09,2.01,0.03,4.84,195.90,13.31,19.76,22.25
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,108.83,-0.10,5.75,1.38,-7.25,-0.35,191.09,-0.07,0.19,10.50


In [68]:
forecast_consumption = pd.read_csv('Forecasted_consumption_202301010000_202503050000_Quarterhour.csv', on_bad_lines='skip', delimiter=';', low_memory=False)

In [69]:
forecast_consumption['Start date'] = pd.to_datetime(forecast_consumption['Start date'], format='%b %d, %Y %I:%M %p')
forecast_consumption['End date'] = pd.to_datetime(forecast_consumption['End date'], format='%b %d, %Y %I:%M %p')

In [70]:
forecast_consumption['hour'] = forecast_consumption['Start date'].dt.hour
forecast_consumption['day'] = forecast_consumption['Start date'].dt.day
forecast_consumption['week'] = forecast_consumption['Start date'].dt.isocalendar().week
forecast_consumption['weekday'] = forecast_consumption['Start date'].dt.weekday  # 0=Monday, 6=Sunday
forecast_consumption['month'] = forecast_consumption['Start date'].dt.month
forecast_consumption['year'] = forecast_consumption['Start date'].dt.year

In [71]:
columns = [
   'Total (grid load) [MWh] Original resolutions', 
    'Residual load [MWh] Original resolutions'
]

forecast_consumption[columns]  = forecast_consumption[columns].replace({'-': np.nan, ',': ''}, regex=True).apply(pd.to_numeric,errors='coerce')

In [72]:
forecast_consumption.head()

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,hour,day,week,weekday,month,year
0,2023-01-01 00:00:00,2023-01-01 00:15:00,10652.25,1091.25,0,1,52,6,1,2023
1,2023-01-01 00:15:00,2023-01-01 00:30:00,10538.00,703.50,0,1,52,6,1,2023
2,2023-01-01 00:30:00,2023-01-01 00:45:00,10380.00,565.75,0,1,52,6,1,2023
3,2023-01-01 00:45:00,2023-01-01 01:00:00,10222.25,438.25,0,1,52,6,1,2023
4,2023-01-01 01:00:00,2023-01-01 01:15:00,10107.75,393.75,1,1,52,6,1,2023


In [ ]:
forecast_consumption.rename(columns=
    {
    'Total (grid load) [MWh] Original resolutions': 'Forecasted_consumption_Total (grid load) [MWh]',
    'Residual load [MWh] Original resolutions': 'Forecasted_consumption_Residual load [MWh]'
    }, 
    inplace=True)

In [74]:
forecast_consumption.head()

,Start date,End date,Forecasted_consumption_Total (grid load) [MWh],Forecasted_consumption_Residual load [MWh],hour,day,week,weekday,month,year
0,2023-01-01 00:00:00,2023-01-01 00:15:00,10652.25,1091.25,0,1,52,6,1,2023
1,2023-01-01 00:15:00,2023-01-01 00:30:00,10538.00,703.50,0,1,52,6,1,2023
2,2023-01-01 00:30:00,2023-01-01 00:45:00,10380.00,565.75,0,1,52,6,1,2023
3,2023-01-01 00:45:00,2023-01-01 01:00:00,10222.25,438.25,0,1,52,6,1,2023
4,2023-01-01 01:00:00,2023-01-01 01:15:00,10107.75,393.75,1,1,52,6,1,2023


In [75]:
merged_df8 = pd.merge(
    merged_df7, 
    forecast_consumption, 
    on=['Start date', 'End date', 'hour', 'day', 'weekday', 'week', 'month', 'year'], 
    how='inner'
)

merged_df8.head()

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Poland [€/MWh] Original resolutions,Sweden 4 [€/MWh] Original resolutions,Switzerland [€/MWh] Original resolutions,Czech Republic [€/MWh] Original resolutions,Northern Italy [€/MWh] Original resolutions,Slovenia [€/MWh] Original resolutions,Hungary [€/MWh] Original resolutions,Austria [MWh] Original resolutions,Forecasted_consumption_Total (grid load) [MWh],Forecasted_consumption_Residual load [MWh]
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,18.09,2.01,0.03,4.84,195.90,13.31,19.76,57.50,10652.25,1091.25
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,18.09,2.01,0.03,4.84,195.90,13.31,19.76,23.25,10538.00,703.50
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,18.09,2.01,0.03,4.84,195.90,13.31,19.76,28.75,10380.00,565.75
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,18.09,2.01,0.03,4.84,195.90,13.31,19.76,22.25,10222.25,438.25
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,5.75,1.38,-7.25,-0.35,191.09,-0.07,0.19,10.50,10107.75,393.75


In [76]:
forecast_generation = pd.read_csv("Forecasted_generation_Day-Ahead_202301010000_202503050000_Hour_Quarterhour.csv", 
                                                      on_bad_lines='skip', delimiter=';', low_memory=False)

In [77]:
forecast_generation['Start date'] = pd.to_datetime(forecast_generation['Start date'], format='%b %d, %Y %I:%M %p')
forecast_generation['End date'] = pd.to_datetime(forecast_generation['End date'], format='%b %d, %Y %I:%M %p')

In [78]:
forecast_generation['hour'] = forecast_generation['Start date'].dt.hour
forecast_generation['day'] = forecast_generation['Start date'].dt.day
forecast_generation['week'] = forecast_generation['Start date'].dt.isocalendar().week
forecast_generation['weekday'] = forecast_generation['Start date'].dt.weekday 
forecast_generation['month'] = forecast_generation['Start date'].dt.month
forecast_generation['year'] = forecast_generation['Start date'].dt.year

In [79]:
columns = [
    "Total [MWh] Original resolutions",
    "Photovoltaics and wind [MWh] Original resolutions",
    "Wind offshore [MWh] Original resolutions",
    "Wind onshore [MWh] Original resolutions",
    "Photovoltaics [MWh] Original resolutions",
    "Other [MWh] Original resolutions"
]

forecast_generation[columns] = forecast_generation[columns].replace({'-': np.nan, ',': ''}, regex=True).apply(pd.to_numeric, errors='coerce')

In [80]:
forecast_generation.head()

,Start date,End date,Total [MWh] Original resolutions,Photovoltaics and wind [MWh] Original resolutions,Wind offshore [MWh] Original resolutions,Wind onshore [MWh] Original resolutions,Photovoltaics [MWh] Original resolutions,Other [MWh] Original resolutions,hour,day,week,weekday,month,year
0,2023-01-01 00:00:00,2023-01-01 00:15:00,56012.0,9561.00,867.25,8693.75,0.0,17018.25,0,1,52,6,1,2023
1,2023-01-01 00:15:00,2023-01-01 00:30:00,NaN,9834.50,869.25,8965.25,0.0,NaN,0,1,52,6,1,2023
2,2023-01-01 00:30:00,2023-01-01 00:45:00,NaN,9814.25,870.25,8944.00,0.0,NaN,0,1,52,6,1,2023
3,2023-01-01 00:45:00,2023-01-01 01:00:00,NaN,9784.00,871.50,8912.50,0.0,NaN,0,1,52,6,1,2023
4,2023-01-01 01:00:00,2023-01-01 01:15:00,56152.0,9714.00,845.50,8868.50,0.0,17417.25,1,1,52,6,1,2023


In [81]:
forecast_generation.rename(
    columns={
        'Total [MWh] Original resolutions': 'Forecasted_generation_Total [MWh]',
        'Photovoltaics and wind [MWh] Original resolutions': 'Forecasted_generation_Photovoltaics and wind [MWh]',
        'Wind offshore [MWh] Original resolutions': 'Forecasted_generation_Wind offshore [MWh]',
        'Wind onshore [MWh] Original resolutions': 'Forecasted_generation_Wind onshore [MWh]',
        'Photovoltaics [MWh] Original resolutions': 'Forecasted_generation_Photovoltaics [MWh]',
        'Other [MWh] Original resolutions': 'Forecasted_generation_Other [MWh]'
    }, inplace=True
)

In [82]:
merged_df9 = pd.merge(
    merged_df8, 
    forecast_generation, 
    on=['Start date', 'End date', 'hour', 'day', 'weekday', 'week', 'month', 'year'], 
    how='inner'
)

merged_df9.head()

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Hungary [€/MWh] Original resolutions,Austria [MWh] Original resolutions,Forecasted_consumption_Total (grid load) [MWh],Forecasted_consumption_Residual load [MWh],Forecasted_generation_Total [MWh],Forecasted_generation_Photovoltaics and wind [MWh],Forecasted_generation_Wind offshore [MWh],Forecasted_generation_Wind onshore [MWh],Forecasted_generation_Photovoltaics [MWh],Forecasted_generation_Other [MWh]
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,19.76,57.50,10652.25,1091.25,56012.0,9561.00,867.25,8693.75,0.0,17018.25
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,19.76,23.25,10538.00,703.50,NaN,9834.50,869.25,8965.25,0.0,NaN
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,19.76,28.75,10380.00,565.75,NaN,9814.25,870.25,8944.00,0.0,NaN
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,19.76,22.25,10222.25,438.25,NaN,9784.00,871.50,8912.50,0.0,NaN
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,0.19,10.50,10107.75,393.75,56152.0,9714.00,845.50,8868.50,0.0,17417.25


In [83]:
frequency_containment = pd.read_csv("Frequency_Containment_Reserve_202301010000_202503050000_Quarterhour.csv", 
                                                      on_bad_lines='skip', delimiter=';', low_memory=False)

In [84]:
frequency_containment['Start date'] = pd.to_datetime(frequency_containment['Start date'], format='%b %d, %Y %I:%M %p')
frequency_containment['End date'] = pd.to_datetime(frequency_containment['End date'], format='%b %d, %Y %I:%M %p')

frequency_containment['hour'] = frequency_containment['Start date'].dt.hour
frequency_containment['day'] = frequency_containment['Start date'].dt.day
frequency_containment['week'] = frequency_containment['Start date'].dt.isocalendar().week
frequency_containment['weekday'] = frequency_containment['Start date'].dt.weekday  
frequency_containment['month'] = frequency_containment['Start date'].dt.month
frequency_containment['year'] = frequency_containment['Start date'].dt.year

In [85]:
columns = [
    "Volume procured [MW] Original resolutions",
    "Procurement price [€/MW] Original resolutions"
]

frequency_containment[columns] = frequency_containment[columns].replace({'-': np.nan, ',': ''}, regex=True).apply(pd.to_numeric,errors='coerce')

C:\Users\GAURAV BHATIYA\AppData\Local\Temp\ipykernel_14880\1973317138.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  frequency_containment[columns] = frequency_containment[columns].replace({'-': np.nan, ',': ''}, regex=True).apply(pd.to_numeric,errors='coerce')


In [86]:
frequency_containment.drop(columns=['Volume procured [MW] Original resolutions'],inplace=True)


In [ ]:
frequency_containment.rename(columns={'Procurement price [€/MW] Original resolutions': 'Frequency_Containment_Reserve_Procurement price [€/MW]'}, inplace=True)

In [88]:
merged_df10 = pd.merge(
    merged_df9, 
    frequency_containment, 
    on=['Start date', 'End date', 'hour', 'day', 'weekday', 'week', 'month', 'year'], 
    how='inner'
)

merged_df10.head()

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Austria [MWh] Original resolutions,Forecasted_consumption_Total (grid load) [MWh],Forecasted_consumption_Residual load [MWh],Forecasted_generation_Total [MWh],Forecasted_generation_Photovoltaics and wind [MWh],Forecasted_generation_Wind offshore [MWh],Forecasted_generation_Wind onshore [MWh],Forecasted_generation_Photovoltaics [MWh],Forecasted_generation_Other [MWh],Frequency_Containment_Reserve_Procurement price [€/MW]
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,57.50,10652.25,1091.25,56012.0,9561.00,867.25,8693.75,0.0,17018.25,8.71
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,23.25,10538.00,703.50,NaN,9834.50,869.25,8965.25,0.0,NaN,8.71
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,28.75,10380.00,565.75,NaN,9814.25,870.25,8944.00,0.0,NaN,8.71
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,22.25,10222.25,438.25,NaN,9784.00,871.50,8912.50,0.0,NaN,8.71
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,10.50,10107.75,393.75,56152.0,9714.00,845.50,8868.50,0.0,17417.25,8.71


In [89]:
generation_forecast = pd.read_csv("Generation_Forecast_Intraday_202301010000_202503050000_Quarterhour.csv", 
                                                      on_bad_lines='skip', delimiter=';', low_memory=False)

In [90]:
generation_forecast['Start date'] = pd.to_datetime(generation_forecast['Start date'], format='%b %d, %Y %I:%M %p')
generation_forecast['End date'] = pd.to_datetime(generation_forecast['End date'], format='%b %d, %Y %I:%M %p')

generation_forecast['hour'] = generation_forecast['Start date'].dt.hour
generation_forecast['day'] = generation_forecast['Start date'].dt.day
generation_forecast['week'] = generation_forecast['Start date'].dt.isocalendar().week
generation_forecast['weekday'] = generation_forecast['Start date'].dt.weekday  
generation_forecast['month'] = generation_forecast['Start date'].dt.month
generation_forecast['year'] = generation_forecast['Start date'].dt.year

In [91]:
columns = [
    "Photovoltaics and wind [MWh] Original resolutions",
    "Wind offshore [MWh] Original resolutions",
    "Wind onshore [MWh] Original resolutions",
    "Photovoltaics [MWh] Original resolutions"
]

generation_forecast[columns] = generation_forecast[columns].replace({'-': np.nan, ',': ''}, regex=True).apply(pd.to_numeric,errors='coerce')

In [92]:
generation_forecast.rename(
    columns={
        'Photovoltaics and wind [MWh] Original resolutions': 'Generation_Forecast_Intraday_Photovoltaics and wind [MWh]',
        'Wind offshore [MWh] Original resolutions': 'Generation_Forecast_Intraday_Wind offshore [MWh]',
        'Wind onshore [MWh] Original resolutions': 'Generation_Forecast_Intraday_Wind onshore [MWh]',
        'Wind onshore [MWh] Original resolutions': 'Forecasted_generation_Wind onshore [MWh]',
        'Photovoltaics [MWh] Original resolutions': 'Generation_Forecast_Intraday_Photovoltaics [MWh]'
    }, inplace=True
)

In [93]:
merged_df11 = pd.merge(
    merged_df10, 
    generation_forecast, 
    on=['Start date', 'End date', 'hour', 'day', 'weekday', 'week', 'month', 'year'], 
    how='inner'
)

merged_df11.head()

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Forecasted_generation_Photovoltaics and wind [MWh],Forecasted_generation_Wind offshore [MWh],Forecasted_generation_Wind onshore [MWh]_x,Forecasted_generation_Photovoltaics [MWh],Forecasted_generation_Other [MWh],Frequency_Containment_Reserve_Procurement price [€/MW],Generation_Forecast_Intraday_Photovoltaics and wind [MWh],Generation_Forecast_Intraday_Wind offshore [MWh],Forecasted_generation_Wind onshore [MWh]_y,Generation_Forecast_Intraday_Photovoltaics [MWh]
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,9561.00,867.25,8693.75,0.0,17018.25,8.71,8419.50,608.75,7810.75,0.0
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,9834.50,869.25,8965.25,0.0,NaN,8.71,8119.25,590.25,7529.00,0.0
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,9814.25,870.25,8944.00,0.0,NaN,8.71,7909.50,597.00,7312.50,0.0
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,9784.00,871.50,8912.50,0.0,NaN,8.71,7961.50,552.75,7408.75,0.0
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,9714.00,845.50,8868.50,0.0,17417.25,8.71,7748.25,535.25,7213.00,0.0


In [94]:
imported_balance =  pd.read_csv("Imported_balancing_services_202301010000_202503050000_Quarterhour.csv", 
                                                      on_bad_lines='skip', delimiter=';', low_memory=False)

In [ ]:
imported_balance['Start date'] = pd.to_datetime(imported_balance['Start date'], format='%b %d, %Y %I:%M %p', errors='coerce')
imported_balance['End date'] = pd.to_datetime(imported_balance['End date'], format='%b %d, %Y %I:%M %p', errors='coerce')

imported_balance['hour'] = imported_balance['Start date'].dt.hour
imported_balance['day'] = imported_balance['Start date'].dt.day
imported_balance['week'] = imported_balance['Start date'].dt.isocalendar().week
imported_balance['weekday'] = imported_balance['Start date'].dt.weekday  
imported_balance['month'] = imported_balance['Start date'].dt.month
imported_balance['year'] = imported_balance['Start date'].dt.year

In [96]:
imported_balance["Austria [MWh] Original resolutions"] = pd.to_numeric(
    imported_balance["Austria [MWh] Original resolutions"], errors='coerce'
)

In [97]:
imported_balance.rename(columns={'Austria [MWh] Original resolutions': 'Imported_balancing_services_Austria [MWh]'}, 
                                    inplace=True)

In [98]:
merged_df12 = pd.merge(
    merged_df11, 
    imported_balance, 
    on=['Start date', 'End date', 'hour', 'day', 'weekday', 'week', 'month', 'year'], 
    how='inner'
)


merged_df12.head()

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Forecasted_generation_Wind offshore [MWh],Forecasted_generation_Wind onshore [MWh]_x,Forecasted_generation_Photovoltaics [MWh],Forecasted_generation_Other [MWh],Frequency_Containment_Reserve_Procurement price [€/MW],Generation_Forecast_Intraday_Photovoltaics and wind [MWh],Generation_Forecast_Intraday_Wind offshore [MWh],Forecasted_generation_Wind onshore [MWh]_y,Generation_Forecast_Intraday_Photovoltaics [MWh],Imported_balancing_services_Austria [MWh]
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,867.25,8693.75,0.0,17018.25,8.71,8419.50,608.75,7810.75,0.0,0.25
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,869.25,8965.25,0.0,NaN,8.71,8119.25,590.25,7529.00,0.0,0.00
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,870.25,8944.00,0.0,NaN,8.71,7909.50,597.00,7312.50,0.0,0.00
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,871.50,8912.50,0.0,NaN,8.71,7961.50,552.75,7408.75,0.0,0.00
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,845.50,8868.50,0.0,17417.25,8.71,7748.25,535.25,7213.00,0.0,0.00


In [99]:
installed_generation = pd.read_csv("Installed_generation_capacity_202301010000_202503050000_Year.csv", 
                                                      on_bad_lines='skip', delimiter=';', low_memory=False)

In [100]:
installed_generation["Start date"] = pd.to_datetime(installed_generation["Start date"], format="%b %d, %Y", errors="coerce")
installed_generation["End date"] = pd.to_datetime(installed_generation["End date"], format="%b %d, %Y", errors="coerce")

installed_generation['hour'] = installed_generation['Start date'].dt.hour
installed_generation['day'] = installed_generation['Start date'].dt.day
installed_generation['week'] = installed_generation['Start date'].dt.isocalendar().week
installed_generation['weekday'] = installed_generation['Start date'].dt.weekday  
installed_generation['month'] = installed_generation['Start date'].dt.month
installed_generation['year'] = installed_generation['Start date'].dt.year

In [101]:
installed_generation.head()

,Start date,End date,Biomass [MW] Original resolutions,Hydropower [MW] Original resolutions,Wind offshore [MW] Original resolutions,Wind onshore [MW] Original resolutions,Photovoltaics [MW] Original resolutions,Other renewable [MW] Original resolutions,Nuclear [MW] Original resolutions,Lignite [MW] Original resolutions,Hard coal [MW] Original resolutions,Fossil gas [MW] Original resolutions,Hydro pumped storage [MW] Original resolutions,Other conventional [MW] Original resolutions,hour,day,week,weekday,month,year
0,2023-01-01,2024-01-01,"8,467.00","5,049.00","8,129.00","57,590.00","63,066.00",440.0,"4,056.00","17,692.00","18,559.00","31,808.00","9,379.00","8,958.00",0,1,52,6,1,2023
1,2024-01-01,2025-01-01,"8,577.00","5,112.00","8,456.00","59,841.00","76,601.00",410.0,-,"18,363.00","18,411.00","36,255.00","9,449.00","8,813.00",0,1,1,0,1,2024
2,2025-01-01,2026-01-01,"8,766.00","5,350.00","9,215.00","63,192.00","86,408.00",446.0,-,"15,176.00","17,509.00","36,614.00","9,384.00","12,971.00",0,1,1,2,1,2025


In [102]:
columns = installed_generation.columns[2:14]

In [103]:
installed_generation[columns] = installed_generation[columns].replace({'-': np.nan, ',': ''}, regex=True).apply(pd.to_numeric,errors='coerce')


In [104]:
installed_generation.rename(
    columns={
        'Biomass [MW] Original resolutions': 'Installed_generation_capacity_Biomass [MW]',
        'Hydropower [MW] Original resolutions': 'Installed_generation_capacity_Hydropower [MW]',
        'Wind offshore [MW] Original resolutions': 'Installed_generation_capacity_Wind offshore [MW]',
        'Wind onshore [MW] Original resolutions': 'Installed_generation_capacity_Wind onshore [MW]',
        'Photovoltaics [MW] Original resolutions': 'Installed_generation_capacity_Photovoltaics [MW]',
        'Other renewable [MW] Original resolutions': 'Installed_generation_capacity_Other renewable [MW]',
        'Nuclear [MW] Original resolutions': 'Installed_generation_capacity_Nuclear [MW]',
        'Lignite [MW] Original resolutions': 'Installed_generation_capacity_Lignite [MW]',
        'Fossil gas [MW] Original resolutions': 'Installed_generation_capacity_Fossil gas [MW]',
        'Hydro pumped storage [MW] Original resolutions': 'Installed_generation_capacity_Hydro pumped storage [MW]',
        'Other conventional [MW] Original resolutions': 'Installed_generation_capacity_Other conventional [MW]'
    }, inplace=True
)

In [105]:
merged_df13 = pd.merge(
    merged_df12, 
    installed_generation, 
    on=['hour', 'day', 'weekday', 'week', 'month', 'year'], 
    how='left'
)

merged_df13.head()

,Start date_x,End date_x,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Installed_generation_capacity_Wind offshore [MW],Installed_generation_capacity_Wind onshore [MW],Installed_generation_capacity_Photovoltaics [MW],Installed_generation_capacity_Other renewable [MW],Installed_generation_capacity_Nuclear [MW],Installed_generation_capacity_Lignite [MW],Hard coal [MW] Original resolutions,Installed_generation_capacity_Fossil gas [MW],Installed_generation_capacity_Hydro pumped storage [MW],Installed_generation_capacity_Other conventional [MW]
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,8129.0,57590.0,63066.0,440.0,4056.0,17692.0,18559.0,31808.0,9379.0,8958.0
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,8129.0,57590.0,63066.0,440.0,4056.0,17692.0,18559.0,31808.0,9379.0,8958.0
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,8129.0,57590.0,63066.0,440.0,4056.0,17692.0,18559.0,31808.0,9379.0,8958.0
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,8129.0,57590.0,63066.0,440.0,4056.0,17692.0,18559.0,31808.0,9379.0,8958.0
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [106]:
merged_df13.rename(columns={'Start date_x': 'Start date', 'End date_x': 'End date'}, inplace=True)
merged_df13.drop(columns=['Start date_y', 'End date_y'], inplace=True)


In [107]:
manual_frequency = pd.read_csv("Manual_Frequency_Restoration_Reserve_202301010000_202503050000_Quarterhour.csv", 
                                                      on_bad_lines='skip', delimiter=';', low_memory=False)

In [108]:
columns = manual_frequency.columns[2:10]

In [109]:
manual_frequency[columns] = manual_frequency[columns].replace({'-': np.nan, ',': ''}, regex=True).apply(pd.to_numeric,errors='coerce')

C:\Users\GAURAV BHATIYA\AppData\Local\Temp\ipykernel_14880\4259330740.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  manual_frequency[columns] = manual_frequency[columns].replace({'-': np.nan, ',': ''}, regex=True).apply(pd.to_numeric,errors='coerce')


In [110]:
manual_frequency['Start date'] = pd.to_datetime(manual_frequency['Start date'], format='%b %d, %Y %I:%M %p')
manual_frequency['End date'] = pd.to_datetime(manual_frequency['End date'], format='%b %d, %Y %I:%M %p')

manual_frequency['hour'] = manual_frequency['Start date'].dt.hour
manual_frequency['day'] = manual_frequency['Start date'].dt.day
manual_frequency['week'] = manual_frequency['Start date'].dt.isocalendar().week
manual_frequency['weekday'] = manual_frequency['Start date'].dt.weekday 
manual_frequency['month'] = manual_frequency['Start date'].dt.month
manual_frequency['year'] = manual_frequency['Start date'].dt.year

In [111]:
drop = [
    'Volume activated (+) [MWh] Original resolutions',
    'Volume activated (-) [MWh] Original resolutions',
    'Activation price (+) [€/MWh] Original resolutions',
    'Activation price (-) [€/MWh] Original resolutions'
]

manual_frequency.drop(columns=drop,inplace=True)

In [112]:
manual_frequency.rename(
    columns={
        'Volume procured (+) [MW] Original resolutions': 'Manual_Frequency_Restoration_Reserve_Volume procured (+) [MW]',
        'Volume procured (-) [MW] Original resolutions': 'Manual_Frequency_Restoration_Reserve_Volume procured (-) [MW]',
        'Procurement price (+) [€/MW] Original resolutions': 'Manual_Frequency_Restoration_Reserve_Procurement price (+) [€/MW]',
        'Procurement price (-) [€/MW] Original resolutions': 'Manual_Frequency_Restoration_Reserve_Procurement price (-) [€/MW]',
    }, inplace=True
)

In [113]:
merged_df14 = pd.merge(
    merged_df13, 
    manual_frequency, 
    on=['Start date', 'End date', 'hour', 'day', 'weekday', 'week', 'month', 'year'], 
    how='inner'
)

merged_df14.head()

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Installed_generation_capacity_Nuclear [MW],Installed_generation_capacity_Lignite [MW],Hard coal [MW] Original resolutions,Installed_generation_capacity_Fossil gas [MW],Installed_generation_capacity_Hydro pumped storage [MW],Installed_generation_capacity_Other conventional [MW],Manual_Frequency_Restoration_Reserve_Volume procured (+) [MW],Manual_Frequency_Restoration_Reserve_Volume procured (-) [MW],Manual_Frequency_Restoration_Reserve_Procurement price (+) [€/MW],Manual_Frequency_Restoration_Reserve_Procurement price (-) [€/MW]
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,4056.0,17692.0,18559.0,31808.0,9379.0,8958.0,977.0,358.0,0.06,1.66
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,4056.0,17692.0,18559.0,31808.0,9379.0,8958.0,977.0,355.0,0.06,1.66
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,4056.0,17692.0,18559.0,31808.0,9379.0,8958.0,977.0,355.0,0.06,1.66
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,4056.0,17692.0,18559.0,31808.0,9379.0,8958.0,977.0,355.0,0.06,1.66
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,NaN,NaN,NaN,NaN,NaN,NaN,977.0,355.0,0.06,1.66


In [114]:
scheduled_commercial = pd.read_csv("Scheduled_commercial_exchanges_202301010000_202503050000_Quarterhour.csv", 
                                                      on_bad_lines='skip', delimiter=';', low_memory=False)

In [115]:
scheduled_commercial['Start date'] = pd.to_datetime(scheduled_commercial['Start date'], format='%b %d, %Y %I:%M %p')
scheduled_commercial['End date'] = pd.to_datetime(scheduled_commercial['End date'], format='%b %d, %Y %I:%M %p')

In [116]:
num_cols = scheduled_commercial.columns.difference(['Start date', 'End date'])

In [118]:
scheduled_commercial['hour'] = scheduled_commercial['Start date'].dt.hour
scheduled_commercial['day'] = scheduled_commercial['Start date'].dt.day
scheduled_commercial['week'] = scheduled_commercial['Start date'].dt.isocalendar().week
scheduled_commercial['weekday'] = scheduled_commercial['Start date'].dt.weekday  # 0=Monday, 6=Sunday
scheduled_commercial['month'] = scheduled_commercial['Start date'].dt.month
scheduled_commercial['year'] = scheduled_commercial['Start date'].dt.year

In [119]:
scheduled_commercial.head()

,Start date,End date,Net export [MWh] Original resolutions,Netherlands (export) [MWh] Original resolutions,Netherlands (import) [MWh] Original resolutions,Switzerland (export) [MWh] Original resolutions,Switzerland (import) [MWh] Original resolutions,Denmark (export) [MWh] Original resolutions,Denmark (import) [MWh] Original resolutions,Czech Republic (export) [MWh] Original resolutions,...,Norway (export) [MWh] Original resolutions,Norway (import) [MWh] Original resolutions,Belgium (export) [MWh] Original resolutions,Belgium (import) [MWh] Original resolutions,hour,day,week,weekday,month,year
0,2023-01-01 00:00:00,2023-01-01 00:15:00,3161.29,378.80,-165.75,197.99,0.0,767.00,-92.98,15.50,...,322.25,0.0,112.90,0.00,0,1,52,6,1,2023
1,2023-01-01 00:15:00,2023-01-01 00:30:00,3135.99,378.80,-165.75,197.99,0.0,767.00,-92.98,15.50,...,322.25,0.0,87.60,0.00,0,1,52,6,1,2023
2,2023-01-01 00:30:00,2023-01-01 00:45:00,3131.54,378.80,-165.75,197.99,0.0,767.00,-92.98,15.50,...,322.25,0.0,83.15,0.00,0,1,52,6,1,2023
3,2023-01-01 00:45:00,2023-01-01 01:00:00,3128.84,378.80,-165.75,197.99,0.0,767.00,-92.98,15.50,...,322.25,0.0,80.45,0.00,0,1,52,6,1,2023
4,2023-01-01 01:00:00,2023-01-01 01:15:00,3852.52,501.15,-95.00,199.81,0.0,751.15,-108.90,177.38,...,322.25,0.0,131.65,-501.63,1,1,52,6,1,2023


In [ ]:
def rename(col):
    
    if col in ['Start date', 'End date', 'hour', 'day', 'week', 'weekday', 'month', 'year']:
        return col
    return f"Scheduled_commercial_exchanges_{col.replace(' Original resolutions', '')}"

In [121]:
scheduled_commercial.rename(columns=rename, inplace=True)

In [122]:
merged_df15 = pd.merge(
    merged_df14, 
    scheduled_commercial, 
    on=['Start date', 'End date', 'hour', 'day', 'weekday', 'week', 'month', 'year'], 
    how='inner'
)

merged_df15.head()

,Start date,End date,Total (grid load) [MWh] Original resolutions,Residual load [MWh] Original resolutions,Hydro pumped storage [MWh] Original resolutions_x,hour,day,weekday,week,month,...,Scheduled_commercial_exchanges_Austria (export) [MWh],Scheduled_commercial_exchanges_Austria (import) [MWh],Scheduled_commercial_exchanges_France (export) [MWh],Scheduled_commercial_exchanges_France (import) [MWh],Scheduled_commercial_exchanges_Poland (export) [MWh],Scheduled_commercial_exchanges_Poland (import) [MWh],Scheduled_commercial_exchanges_Norway (export) [MWh],Scheduled_commercial_exchanges_Norway (import) [MWh],Scheduled_commercial_exchanges_Belgium (export) [MWh],Scheduled_commercial_exchanges_Belgium (import) [MWh]
0,2023-01-01 00:00:00,2023-01-01 00:15:00,9673.00,1842.5,494.00,0,1,6,52,1,...,547.03,0.0,859.99,0.0,6.43,-1.43,322.25,0.0,112.90,0.00
1,2023-01-01 00:15:00,2023-01-01 00:30:00,9593.50,1691.5,502.50,0,1,6,52,1,...,547.03,0.0,859.99,0.0,6.43,-1.43,322.25,0.0,87.60,0.00
2,2023-01-01 00:30:00,2023-01-01 00:45:00,9562.00,1442.5,561.00,0,1,6,52,1,...,547.03,0.0,859.99,0.0,6.43,-1.43,322.25,0.0,83.15,0.00
3,2023-01-01 00:45:00,2023-01-01 01:00:00,9517.50,1598.5,519.25,0,1,6,52,1,...,547.03,0.0,859.99,0.0,6.43,-1.43,322.25,0.0,80.45,0.00
4,2023-01-01 01:00:00,2023-01-01 01:15:00,9433.25,1325.5,301.00,1,1,6,52,1,...,723.36,0.0,1368.55,0.0,189.40,-5.88,322.25,0.0,131.65,-501.63


In [123]:
merged_df15.to_csv('merged_df15.csv', index=False)